# LAB 6 — Análise de Erros de Faithfulness
## Aula 5 — Avaliação de Pipelines RAG

Identifica queries com **Faithfulness < 0.70** no baseline do LAB 2, **diagnostica a causa** via Groq/Ollama e aplica **3 correções** (rerank, query rewrite, expansão de contexto), recalculando o Faithfulness.

**Stack:** Groq `llama-3.1-8b-instant` (fallback Ollama) + OllamaEmbeddings BGE-M3 + OpenSearch 3.x.


## 1. Instalação + Stack


In [ ]:
import subprocess, sys
for pkg in ['ragas>=0.1.16','datasets','pandas','matplotlib','tqdm',
            'langchain>=0.2','langchain-core>=0.2',
            'langchain-groq>=0.1','langchain-ollama>=0.1',
            'opensearch-py>=2.7','python-dotenv>=1.0']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
print('dependencias instaladas')


In [ ]:
import os, json, time, warnings, math
from pathlib import Path
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

for ep in [Path.cwd()/'.env', Path.home()/'mba-rag'/'.env',
            Path.cwd().parent.parent/'.env']:
    if ep.exists(): load_dotenv(ep, override=True); break

GROQ_API_KEY       = os.getenv('GROQ_API_KEY','')
GROQ_LLM_MODEL     = os.getenv('GROQ_LLM_MODEL','llama-3.1-8b-instant')
OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL','http://localhost:11434')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL','llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL','bge-m3')
OPENSEARCH_HOST    = os.getenv('OPENSEARCH_HOST','localhost')
OPENSEARCH_PORT    = int(os.getenv('OPENSEARCH_PORT',9200))
OPENSEARCH_USER    = os.getenv('OPENSEARCH_USER','admin')
OPENSEARCH_PASS    = os.getenv('OPENSEARCH_PASSWORD','admin')
INDEX_NAME         = os.getenv('INDEX_NAME','corpus_juridico_aula4')

FAITH_THR = 0.70
META = {'faithfulness':0.80,'answer_relevancy':0.75,
        'context_recall':0.70,'context_precision':0.70}

from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from opensearchpy import OpenSearch
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

llm = None; LLM_PROVIDER = None; LLM_MODEL = None
if GROQ_API_KEY:
    try:
        c = ChatGroq(model=GROQ_LLM_MODEL, api_key=GROQ_API_KEY,
                     temperature=0.0, max_tokens=512, timeout=30)
        c.invoke([HumanMessage(content='ok')])
        llm, LLM_PROVIDER, LLM_MODEL = c, 'groq', GROQ_LLM_MODEL
    except Exception: pass
if llm is None:
    llm = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL,
                     temperature=0.0, num_predict=512)
    LLM_PROVIDER, LLM_MODEL = 'ollama', OLLAMA_LLM_MODEL

embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
_ = embeddings.embed_query('teste'); assert len(_) == 1024

os_client = OpenSearch(
    hosts=[{'host':OPENSEARCH_HOST,'port':OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER,OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, timeout=60,
)

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
print(f'Stack OK | LLM={LLM_PROVIDER}/{LLM_MODEL} | embed={OLLAMA_EMBED_MODEL}')


## 2. Carregar baseline do LAB2 e filtrar queries com Faithfulness < 0.70


In [ ]:
import pandas as pd
LAB2_CSV = 'ragas_baseline_resultados.csv'
DATASET = 'dataset_avaliacao_completo.json'
if not os.path.exists(LAB2_CSV):
    raise FileNotFoundError(f'{LAB2_CSV} nao encontrado - execute LAB2')
if not os.path.exists(DATASET):
    raise FileNotFoundError(f'{DATASET} nao encontrado - execute LAB1')

df_naive = pd.read_csv(LAB2_CSV)
with open(DATASET, encoding='utf-8') as f: dataset = json.load(f)

# alinha pelos ids quando disponiveis
if 'id' in df_naive.columns:
    ids_baixos = df_naive[df_naive['faithfulness'] < FAITH_THR]['id'].tolist()
else:
    df_naive['id'] = [p.get('id','') for p in dataset[:len(df_naive)]]
    ids_baixos = df_naive[df_naive['faithfulness'] < FAITH_THR]['id'].tolist()

queries_ruins = [p for p in dataset if p['id'] in ids_baixos]
print(f'pares Faithfulness < {FAITH_THR}: {len(queries_ruins)}')
if not queries_ruins:
    print('  Todos com Faith >= 0.70 - sem queries para analisar')
    print('  Vamos analisar os 3 piores ainda assim:')
    worst = df_naive.nsmallest(3, 'faithfulness')['id'].tolist()
    queries_ruins = [p for p in dataset if p['id'] in worst]
    print(f'  3 piores: {worst}')


## 3. Diagnóstico via LLM (Groq/Ollama)

Para cada query ruim, identificamos a causa do baixo Faithfulness em 3 categorias:
- `retrieval_ruim`: chunks recuperados não cobrem a pergunta
- `prompt_vago`: prompt do gerador não cita os trechos explicitamente
- `query_ambígua`: pergunta admite múltiplas interpretações


In [ ]:
DIAG_PROMPT = ChatPromptTemplate.from_messages([
    ('system','Voce e um especialista em sistemas RAG. Diagnostique a causa do baixo Faithfulness '
              'em 3 categorias: retrieval_ruim, prompt_vago, query_ambigua. Responda APENAS uma destas '
              'palavras + uma frase curta de justificativa, no formato:  CATEGORIA: <justificativa>'),
    ('human','PERGUNTA: {pergunta}\n\nRESPOSTA GERADA: {answer}\n\nGROUND TRUTH: {gt}')
])
DIAG_CHAIN = DIAG_PROMPT | llm | StrOutputParser()

def diagnosticar(par: dict, df_row: pd.Series) -> dict:
    try:
        ans = df_row.get('answer', '') or '(sem resposta)'
        diag = DIAG_CHAIN.invoke({
            'pergunta': par['question'][:600],
            'answer': str(ans)[:600],
            'gt': par['ground_truth'][:600],
        }).strip()
        cat = diag.split(':',1)[0].strip().lower()
        if cat not in ['retrieval_ruim','prompt_vago','query_ambigua']:
            cat = 'retrieval_ruim'
        return {'causa': cat, 'diagnostico': diag}
    except Exception as e:
        return {'causa':'retrieval_ruim','diagnostico':f'erro: {e}'}

print('Diagnosticando 3 piores queries...')
piores = sorted(
    [(p, df_naive[df_naive['id']==p['id']].iloc[0])
     for p in queries_ruins if not df_naive[df_naive['id']==p['id']].empty],
    key=lambda x: x[1].get('faithfulness',0))[:3]

diagnosticos = []
for par, row in piores:
    d = diagnosticar(par, row)
    d.update({'id': par['id'], 'question': par['question'],
              'tipo': par.get('tipo',''),
              'faithfulness_naive': float(row.get('faithfulness',0))})
    diagnosticos.append(d)
    print(f'  [{par["id"]}] Faith={row.get("faithfulness",0):.3f} -> {d["causa"]}')
    print(f'    {d["diagnostico"][:160]}...')


## 4. Aplicar Correções e Recalcular Faithfulness

Correções aplicadas conforme a causa:
- `retrieval_ruim` → aumenta k de 5 para 10 e usa hybrid (BM25+kNN)
- `prompt_vago`  → prompt reforçado com instrução de citação obrigatória
- `query_ambigua` → reescrita da query pelo LLM antes do retrieval


In [ ]:
PROMPT_PADRAO = ChatPromptTemplate.from_messages([
    ('system','Voce e um assistente juridico brasileiro. Responda APENAS com base nos trechos.'),
    ('human','TRECHOS:\n{contextos}\n\nPERGUNTA: {pergunta}\n\nRESPOSTA:')
])
PROMPT_REFORCADO = ChatPromptTemplate.from_messages([
    ('system','Voce e um assistente juridico brasileiro. CITE OBRIGATORIAMENTE o numero do trecho '
              'usado em cada afirmacao no formato (Trecho N). NAO invente fatos.'),
    ('human','TRECHOS:\n{contextos}\n\nPERGUNTA: {pergunta}\n\nRESPOSTA (com citacoes):')
])
PROMPT_REWRITE = ChatPromptTemplate.from_messages([
    ('system','Reescreva a pergunta juridica abaixo em uma forma mais clara e especifica, '
              'mantendo o sentido. Responda APENAS a pergunta reescrita.'),
    ('human','{pergunta}')
])

GEN_PADRAO    = PROMPT_PADRAO    | llm | StrOutputParser()
GEN_REFORCADO = PROMPT_REFORCADO | llm | StrOutputParser()
REWRITE_CHAIN = PROMPT_REWRITE   | llm | StrOutputParser()

def retrieve_knn(q: str, k: int = 5):
    v = embeddings.embed_query(q)
    r = os_client.search(index=INDEX_NAME,
                          body={'size':k,'query':{'knn':{'embedding':{'vector':v,'k':k}}},
                                '_source':['titulo','conteudo']})
    return [(h['_source'].get('titulo','')+'\n'+h['_source'].get('conteudo','')).strip()
             for h in r['hits']['hits']]

def retrieve_hybrid(q: str, k: int = 10):
    v = embeddings.embed_query(q)
    try:
        r = os_client.search(index=INDEX_NAME,
            body={'size':k,'query':{'hybrid':{'queries':[
                {'match':{'conteudo':{'query':q}}},
                {'knn':{'embedding':{'vector':v,'k':k}}}]}},
            '_source':['titulo','conteudo']},
            params={'search_pipeline':'hybrid_rrf_pipeline'})
    except Exception:
        r = os_client.search(index=INDEX_NAME,
            body={'size':k,'query':{'bool':{'should':[
                {'match':{'conteudo':{'query':q}}},
                {'knn':{'embedding':{'vector':v,'k':k}}}]}},
            '_source':['titulo','conteudo']})
    return [(h['_source'].get('titulo','')+'\n'+h['_source'].get('conteudo','')).strip()
             for h in r['hits']['hits']]

def aplicar_correcao(par: dict, causa: str) -> dict:
    if causa == 'retrieval_ruim':
        ctxs = retrieve_hybrid(par['question'], k=10)
        chain = GEN_PADRAO; q = par['question']
    elif causa == 'prompt_vago':
        ctxs = retrieve_knn(par['question'], k=5)
        chain = GEN_REFORCADO; q = par['question']
    elif causa == 'query_ambigua':
        q = REWRITE_CHAIN.invoke({'pergunta': par['question']}).strip()
        ctxs = retrieve_knn(q, k=5)
        chain = GEN_PADRAO
    else:
        ctxs = retrieve_knn(par['question'], k=5); chain = GEN_PADRAO; q = par['question']
    bloco = '\n\n'.join(f'[Trecho {i+1}]\n{c[:600]}' for i, c in enumerate(ctxs))
    ans = chain.invoke({'contextos': bloco, 'pergunta': q}).strip()
    return {'question': par['question'], 'q_rewritten': q if q != par['question'] else None,
            'answer': ans, 'contexts': ctxs, 'ground_truth': par['ground_truth']}

print('Aplicando correcoes nas 3 piores queries...')
corrigidas = []
for d in diagnosticos:
    par = next(p for p in queries_ruins if p['id'] == d['id'])
    r = aplicar_correcao(par, d['causa'])
    r.update({'id': d['id'], 'tipo': d['tipo'], 'causa': d['causa']})
    corrigidas.append(r)
    print(f'  [{d["id"]}] causa={d["causa"]} -> answer head: {r["answer"][:120]}...')


## 5. Recalcular Faithfulness das versões corrigidas


In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness

ds_cor = Dataset.from_list([{
    'question': r['question'], 'answer': r['answer'],
    'contexts': r['contexts'], 'ground_truth': r['ground_truth'],
} for r in corrigidas])

print('Calculando RAGAS (Faithfulness) das versoes corrigidas...')
res_cor = evaluate(ds_cor, metrics=[faithfulness],
                    llm=ragas_llm, embeddings=ragas_embeddings,
                    raise_exceptions=False)
df_cor = res_cor.to_pandas()
faith_cor = df_cor['faithfulness'].tolist()
for i, r in enumerate(corrigidas):
    r['faithfulness_corrigido'] = float(faith_cor[i]) if not math.isnan(float(faith_cor[i])) else 0.0

print('\nResultados:')
print(f"{'id':<20}{'causa':<18}{'naive':>8}{'corrigido':>10}{'delta':>9}")
print('-' * 70)
for d, r in zip(diagnosticos, corrigidas):
    n = d['faithfulness_naive']; c = r['faithfulness_corrigido']; delta = c - n
    print(f'{d["id"]:<20}{d["causa"]:<18}{n:>8.4f}{c:>10.4f}{delta:>+9.4f}')


## 6. Tabela Comparativa + Gráfico


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

linhas = []
for d, r in zip(diagnosticos, corrigidas):
    linhas.append({
        'id': d['id'], 'tipo': d['tipo'], 'causa': d['causa'],
        'question': d['question'][:100],
        'diagnostico': d['diagnostico'][:140],
        'faith_naive': round(d['faithfulness_naive'], 4),
        'faith_corrigido': round(r['faithfulness_corrigido'], 4),
        'delta': round(r['faithfulness_corrigido'] - d['faithfulness_naive'], 4),
        'q_rewritten': r.get('q_rewritten'),
    })
df_comp = pd.DataFrame(linhas)
df_comp.to_csv('lab6_analise_erros.csv', index=False, encoding='utf-8')
print(df_comp[['id','causa','faith_naive','faith_corrigido','delta']].to_string(index=False))

# barras horizontais
fig, ax = plt.subplots(figsize=(10, 5))
y = range(len(df_comp))
ax.barh([i-0.2 for i in y], df_comp['faith_naive'],     0.4, color='#e74c3c', alpha=0.85, label='naive')
ax.barh([i+0.2 for i in y], df_comp['faith_corrigido'], 0.4, color='#2ecc71', alpha=0.85, label='corrigido')
ax.axvline(0.80, color='red', linestyle='--', label='meta 0.80')
ax.axvline(0.70, color='orange', linestyle=':', label='limiar análise 0.70')
ax.set_yticks(list(y)); ax.set_yticklabels([f'{r["id"]}\n({r["causa"]})' for _, r in df_comp.iterrows()])
ax.set_xlim(0, 1.05); ax.set_xlabel('Faithfulness')
ax.set_title('Faithfulness: antes vs depois das correcoes', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig('lab6_faithfulness_antes_depois.png', dpi=140); plt.show()
print('\nlab6_analise_erros.csv + lab6_faithfulness_antes_depois.png salvos')


## 7. Manual de Correção Rápida

| Causa identificada       | Correção aplicada                                | Quando funciona             |
|--------------------------|--------------------------------------------------|------------------------------|
| `retrieval_ruim`         | Hybrid Search (BM25 + kNN BGE-M3) + k=10         | Corpus tem o conteúdo mas embedding único não recupera |
| `prompt_vago`            | Prompt reforçado com citação obrigatória         | Modelo divaga; basta forçar grounding explícito |
| `query_ambigua`          | Query rewrite via LLM antes do retrieval         | Pergunta com múltiplas interpretações |

> **Próximo passo:** integrar essas 3 correções ao pipeline padrão como **camada de fallback** quando Faithfulness por par < 0.70 — automatiza o LAB 6 em produção.

## Referências

ES, S. et al. **RAGAS**. arXiv:2309.15217, 2023.  
GAO, L. et al. **Precise Zero-Shot Dense Retrieval without Relevance Labels (HyDE)**. ACL, 2023.  
GROQ INC. **Groq API**. <https://console.groq.com/docs>.
